# Visual Exploration and Interactive Dashboards

## Café sales practice notebook

This notebook follows the supplied practice guide from first inspection through a shareable dashboard and a short manager report. The data is synthetic: one year of daily revenue by café outlet and product category. Revenue is measured in **Rs thousand**.

### Learning path

1. Inspect shape, dates, types, missing values, and group coverage.
2. Explore monthly, weekday, category, and offer patterns.
3. Investigate unusual values and document a defensible cleaning decision.
4. Build interactive charts and a standalone HTML dashboard.
5. Export a filterable Streamlit app and write a concise report.

> Work through the prompts before opening the worked answers near the end. The three extreme values are intentional data-entry slips; other high values can be legitimate busy days. An outlier flag is a prompt to investigate, not proof of an error.


## 0. Setup and reproducibility

This notebook uses NumPy, pandas, Matplotlib, Seaborn, and Plotly. If an import fails, install the missing packages in your notebook environment. Streamlit is only needed if you want to launch the optional app near the end.


In [ ]:
# If needed, install the notebook libraries in the active kernel:
# %pip install numpy pandas matplotlib seaborn plotly

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, HTML

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")


## 1. Build the practice dataset

The fixed seed makes the generated values reproducible. There are 365 days × 3 outlets × 3 items = 3,285 rows. `Revenue` is in Rs thousand.

In [ ]:
# Reproducible daily sales for three outlets and three item categories
np.random.seed(21)
dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
outlets = ["Pune", "Jaipur", "Kochi"]
items = ["Beverages", "Bakery", "Merchandise"]

cafe = pd.DataFrame(
    [(date, outlet, item) for date in dates for outlet in outlets for item in items],
    columns=["Date", "Outlet", "Item"],
)

base = {"Beverages": 300, "Bakery": 180, "Merchandise": 90}
n = len(cafe)
cafe["Offer"] = np.random.choice([0, 5, 10, 15, 20, 25], n)
growth = 1 + cafe["Date"].dt.dayofyear / 365 * 0.25
weekend = np.where(cafe["Date"].dt.dayofweek >= 5, 1.35, 1.0)
winter = np.where(cafe["Date"].dt.month.isin([12, 1]), 1.25, 1.0)
outlet_factor = cafe["Outlet"].map({"Pune": 1.1, "Jaipur": 0.95, "Kochi": 1.0})

cafe["Revenue"] = (
    cafe["Item"].map(base) * growth * weekend * winter * outlet_factor
    * (1 + cafe["Offer"] / 100) * np.random.normal(1, 0.07, n)
).round(1)

# Inject three synthetic data-entry slips for the outlier exercise.
cafe.loc[[300, 1800, 2900], "Revenue"] *= 7

display(cafe.head())
print(f"Rows: {len(cafe):,}; columns: {len(cafe.columns)}")


## 2. First look: validate the table before charting

Check the dataset size, date coverage, types, missingness, numeric ranges, and whether each outlet has the expected number of records.

In [ ]:
print("Shape:", cafe.shape)
print("Date range:", cafe["Date"].min().date(), "to", cafe["Date"].max().date())
print("\nColumn information:")
cafe.info()
display(cafe.describe(include="all").T)
print("\nMissing values per column:")
display(cafe.isna().sum().rename("missing_count").to_frame())
print("\nRows per outlet:")
display(cafe["Outlet"].value_counts().sort_index().rename("rows").to_frame())


### Your turn: first look

In a Markdown cell, state the number of rows, date range, whether anything is missing, and why outlet row counts are equal. Then add the date-part columns below and inspect a few rows.


In [ ]:
cafe["Month"] = cafe["Date"].dt.month
cafe["Weekday"] = cafe["Date"].dt.dayofweek  # Monday=0 ... Sunday=6
cafe.head(3)


## 3. Patterns over time

A line chart suits ordered time; a bar chart compares separate weekday groups. Revenue sums are totals across rows, while weekday averages help compare groups with similar coverage.

In [ ]:
monthly_total = cafe.groupby("Month")["Revenue"].sum()
ax = monthly_total.plot(marker="o", figsize=(10, 4), color="#176B87")
ax.set(title="Total café revenue by month", xlabel="Month (1 = January)", ylabel="Revenue (Rs thousand)")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

display((monthly_total / 1000).round(1).rename("Revenue (Rs million)").to_frame())


In [ ]:
weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday_avg = cafe.groupby("Weekday")["Revenue"].mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(weekday_names, weekday_avg.values, color=["#77AADD"] * 5 + ["#EE8866"] * 2)
ax.set(title="Average revenue per row by weekday", xlabel="Day of week", ylabel="Revenue (Rs thousand)")
plt.tight_layout()
plt.show()

weekend_mean = weekday_avg.loc[[5, 6]].mean()
weekday_mean = weekday_avg.loc[[0, 1, 2, 3, 4]].mean()
print(f"Weekend average: {weekend_mean:.1f} Rs thousand")
print(f"Weekday average: {weekday_mean:.1f} Rs thousand")
print(f"Weekend lift: {(weekend_mean / weekday_mean - 1):.1%}")


### Exercise: item and weekday patterns

1. Draw the monthly total for each item as three lines. Which item grows most from February to December?
2. For Merchandise only, compare average weekend revenue with average weekday revenue. Is its weekend lift similar to all items?


In [ ]:
# Worked analysis for the exercise
by_item_month = cafe.groupby(["Month", "Item"])["Revenue"].sum().unstack()
ax = by_item_month.plot(marker="o", figsize=(10, 5))
ax.set(title="Monthly revenue by item", xlabel="Month (1 = January)", ylabel="Revenue (Rs thousand)")
plt.tight_layout()
plt.show()
display(((by_item_month.loc[12] / by_item_month.loc[2] - 1) * 100).round(1).rename("Feb-to-Dec growth (%)").to_frame())

merch = cafe[cafe["Item"] == "Merchandise"].groupby("Weekday")["Revenue"].mean()
ax = merch.plot(kind="bar", figsize=(8, 4), color="#44AA99")
ax.set(title="Merchandise average revenue by weekday", xlabel="Weekday (0 = Monday)", ylabel="Revenue (Rs thousand)")
ax.set_xticklabels(weekday_names, rotation=0)
plt.tight_layout()
plt.show()
merch_weekend_lift = merch.loc[[5, 6]].mean() / merch.loc[[0, 1, 2, 3, 4]].mean() - 1
print(f"Merchandise weekend lift: {merch_weekend_lift:.1%}")


## 4. Two categories at once: pivot tables and heatmaps

A pivot table gives a compact comparison across two categories. The first grid below compares average revenue by outlet and item.

In [ ]:
grid = cafe.pivot_table(index="Outlet", columns="Item", values="Revenue", aggfunc="mean")
display(grid.round(1))
plt.figure(figsize=(8, 4))
sns.heatmap(grid, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "Revenue (Rs thousand)"})
plt.title("Average revenue by outlet and item")
plt.tight_layout()
plt.show()


### Exercise: month by outlet

Build a Month × Outlet grid of average revenue. Identify the highest cell and check whether outlet ordering remains the same in every month.

In [ ]:
grid_month_outlet = cafe.pivot_table(index="Month", columns="Outlet", values="Revenue", aggfunc="mean")
display(grid_month_outlet.round(1))
plt.figure(figsize=(8, 6))
sns.heatmap(grid_month_outlet, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "Revenue (Rs thousand)"})
plt.title("Average revenue by month and outlet")
plt.tight_layout()
plt.show()
highest_month_outlet = grid_month_outlet.stack().idxmax()
print("Highest cell (month, outlet):", highest_month_outlet, f"= {grid_month_outlet.stack().max():.1f}")


## 5. Does the offer move with revenue?

This is a descriptive comparison. An association between offer level and revenue does **not** establish that the offer caused the difference; item mix, timing, and other factors can vary between offer groups.

In [ ]:
offer_avg = cafe.groupby("Offer")["Revenue"].mean()
ax = offer_avg.plot(kind="bar", figsize=(8, 4), color="#AA3377")
ax.set(title="Average revenue by offer level", xlabel="Offer (%)", ylabel="Revenue (Rs thousand)")
ax.set_xticklabels([str(x) for x in offer_avg.index], rotation=0)
plt.tight_layout()
plt.show()
display(pd.DataFrame({"average_revenue": offer_avg, "step_change": offer_avg.diff()}))


### Exercise: offers for Beverages

Repeat the offer comparison for Beverages only. Is the rise steady? Use the step changes to support your answer.

In [ ]:
bev_offer = cafe.loc[cafe["Item"] == "Beverages"].groupby("Offer")["Revenue"].mean()
ax = bev_offer.plot(kind="bar", figsize=(8, 4), color="#228833")
ax.set(title="Average Beverages revenue by offer", xlabel="Offer (%)", ylabel="Revenue (Rs thousand)")
ax.set_xticklabels([str(x) for x in bev_offer.index], rotation=0)
plt.tight_layout()
plt.show()
display(pd.DataFrame({"average_revenue": bev_offer, "step_change": bev_offer.diff()}))


## 6. Outliers: find, inspect, decide

A box plot flags values far from the middle of each group. A flag means investigate, not automatically delete. Hover the points to identify the date and outlet.

In [ ]:
fig = px.box(cafe, x="Item", y="Revenue", hover_data=["Date", "Outlet", "Offer"],
             title="Revenue distribution by item (Rs thousand)")
fig.show()


### Inspect the largest values

Ask whether each value is plausible, whether its context explains it, and whether it can be checked against a source. The synthetic example has three extreme values that are far larger than their peers.

In [ ]:
largest = cafe.nlargest(10, "Revenue")[["Date", "Outlet", "Item", "Offer", "Revenue"]]
display(largest)


### Cleaning decision

For this synthetic exercise, use a transparent item-specific rule: flag revenue greater than three times that item's overall mean. The guide's three injected slips are removed by this rule. In real work, validate suspect records against source systems and retain an audit trail; never remove an observation solely because it is high.

In [ ]:
item_mean = cafe.groupby("Item")["Revenue"].transform("mean")
outlier_mask = cafe["Revenue"] > 3 * item_mean
flagged = cafe.loc[outlier_mask].copy()
display(flagged[["Date", "Outlet", "Item", "Offer", "Revenue"]])

clean = cafe.loc[~outlier_mask].copy()
print(f"Flagged and excluded for this synthetic exercise: {len(flagged)}")
print(f"Rows retained: {len(clean):,}")


### Exercise: compare a blunt rule

Repeat the same cutoff using each outlet's mean instead of each item's mean. How many rows does it flag? Inspect the additional rows and decide whether the rule is too broad.

In [ ]:
outlet_mean = cafe.groupby("Outlet")["Revenue"].transform("mean")
too_many_mask = cafe["Revenue"] > 3 * outlet_mean
too_many = cafe.loc[too_many_mask].sort_values("Revenue", ascending=False)
print("Rows flagged by outlet rule:", int(too_many_mask.sum()))
display(too_many[["Date", "Outlet", "Item", "Offer", "Revenue"]])


## 7. Interactive Plotly charts

Plotly charts support hover, zoom, and clickable legends. Prepare summary tables first so each chart has a clear measure and unit.

In [ ]:
# Monthly revenue by outlet
monthly = clean.groupby(["Month", "Outlet"], as_index=False)["Revenue"].sum()
fig_a = px.line(monthly, x="Month", y="Revenue", color="Outlet", markers=True,
                title="Monthly revenue by outlet",
                labels={"Month": "Month (1 = January)", "Revenue": "Revenue (Rs thousand)"})

# Total revenue by item
by_item = clean.groupby("Item", as_index=False)["Revenue"].sum()
fig_b = px.bar(by_item, x="Item", y="Revenue", title="Total revenue by item",
               labels={"Revenue": "Revenue (Rs thousand)"})

# Average revenue by outlet and item
grid_clean = clean.pivot_table(index="Outlet", columns="Item", values="Revenue", aggfunc="mean")
fig_c = px.imshow(grid_clean, text_auto=".0f", color_continuous_scale="Blues",
                  title="Average revenue: outlet vs item",
                  labels={"x": "Item", "y": "Outlet", "color": "Rs thousand"})

# Average revenue by offer
by_offer = clean.groupby("Offer", as_index=False)["Revenue"].mean()
fig_d = px.bar(by_offer, x="Offer", y="Revenue", title="Average revenue by offer (%)",
               labels={"Offer": "Offer (%)", "Revenue": "Average revenue (Rs thousand)"})

fig_a.show()


In [ ]:
# Explore the other charts one at a time
fig_b.show()
fig_c.show()
fig_d.show()


## 8. Assemble and save a four-chart dashboard

The saved HTML file is interactive and can be opened in a browser without running Python. It contains the chart data, so expect a larger file than a static image.

In [ ]:
from plotly.subplots import make_subplots

dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Monthly revenue by outlet", "Total revenue by item",
                    "Average revenue: outlet vs item", "Average revenue by offer (%)"),
)
for trace in fig_a.data:
    dashboard.add_trace(trace, row=1, col=1)
for trace in fig_b.data:
    dashboard.add_trace(trace, row=1, col=2)
for trace in fig_c.data:
    dashboard.add_trace(trace, row=2, col=1)
for trace in fig_d.data:
    dashboard.add_trace(trace, row=2, col=2)

dashboard.update_coloraxes(colorscale="Blues", showscale=False)
dashboard.update_xaxes(title_text="Month (1 = January)", row=1, col=1)
dashboard.update_yaxes(title_text="Revenue (Rs thousand)", row=1, col=1)
dashboard.update_xaxes(title_text="Offer (%)", row=2, col=2)
dashboard.update_yaxes(title_text="Average revenue (Rs thousand)", row=2, col=2)
dashboard.update_layout(height=720, title_text="Café Sales Dashboard", template="simple_white")
dashboard.show()

dashboard_path = "cafe_dashboard.html"
dashboard.write_html(dashboard_path, include_plotlyjs=True)
print(f"Saved interactive dashboard: {dashboard_path}")


### Exercise: a two-panel dashboard

Create a 1 × 2 dashboard with total revenue by outlet and monthly revenue for Beverages. Label both axes with units, save it to HTML, and check hover and zoom in the browser.

In [ ]:
# Worked version of the exercise
by_outlet = clean.groupby("Outlet", as_index=False)["Revenue"].sum()
bev_month = clean.loc[clean["Item"] == "Beverages"].groupby("Month", as_index=False)["Revenue"].sum()
fig_outlet = px.bar(by_outlet, x="Outlet", y="Revenue")
fig_bev_month = px.line(bev_month, x="Month", y="Revenue", markers=True)

two_panel = make_subplots(rows=1, cols=2, subplot_titles=("Total revenue by outlet", "Beverages revenue by month"))
for trace in fig_outlet.data:
    two_panel.add_trace(trace, row=1, col=1)
for trace in fig_bev_month.data:
    two_panel.add_trace(trace, row=1, col=2)
two_panel.update_xaxes(title_text="Outlet", row=1, col=1)
two_panel.update_yaxes(title_text="Revenue (Rs thousand)", row=1, col=1)
two_panel.update_xaxes(title_text="Month (1 = January)", row=1, col=2)
two_panel.update_yaxes(title_text="Revenue (Rs thousand)", row=1, col=2)
two_panel.update_layout(height=460, title_text="Outlets and Beverages", template="simple_white", showlegend=False)
two_panel.show()
two_panel.write_html("outlets_and_beverages.html", include_plotlyjs=True)


## 9. Export a filterable Streamlit app

The live dashboard lives in `app.py` next to this notebook. It reloads `cafe_clean.csv` and redraws the numbers when you move a filter. Streamlit is optional — install it only if you want to click around in the app.

This cell writes the cleaned table. The next cell checks that `app.py` is there, and will **not** smash the dashboard file if you already have one.



In [ ]:
from pathlib import Path

clean.to_csv("cafe_clean.csv", index=False)
print("Saved cafe_clean.csv")
print("Rows:", len(clean))


### `app.py`

The dashboard now follows an executive summary layout: five headline metrics, an original inline café illustration, and two key charts side by side. Tabs keep product, outlet, offer, and data views focused. The monthly trend marks its peak month; bars show values directly; hover details include sample counts where useful. The blue, pink, and gold palette follows the supplied visual reference while keeping labels readable.


In [ ]:
from pathlib import Path

app_path = Path("app.py")
if app_path.exists():
    print("Found", app_path.resolve())
    print("Start it with:  streamlit run app.py")
else:
    print("app.py is missing. Put the dashboard script in this folder.")


### Run and explore the app (optional)

From a terminal in this folder:

```bash
python -m pip install -r requirements.txt
streamlit run app.py
```

Use the tabs to explore one question at a time. Adjust filters and compare total revenue with average record values. Bold labels point out what to notice; italic notes explain how to read the chart. The Data tab includes a distribution view and a CSV download. Stop the app with Ctrl+C in the terminal.


## 10. Write a report for a manager

A useful one-page report has a clear title and period, three checkable findings with numbers, a data-quality note, and two actions tied to the findings. Keep language plain and distinguish association from cause.

### Report draft

**Café sales in 2025: what the data shows**  
This report covers daily item revenue at Pune, Jaipur, and Kochi from January through December 2025. Revenue is shown in Rs thousand unless stated otherwise.

**Findings**

- December was the strongest month at about Rs 102.9 million, versus Rs 80.7 million in November (about 28% higher).
- Weekend average revenue was about Rs 346 thousand per row, compared with Rs 257 thousand on weekdays (about 35% higher).
- Pune led the average for every item; its Beverages average was about Rs 488 thousand, compared with Rs 438 thousand in Kochi and Rs 415 thousand in Jaipur.

**Data quality**  
Three unusually large synthetic entries were excluded after comparing each row with its item group. The decision is specific to this practice dataset; in operational reporting, verify questionable entries against source records. The offer comparison is observational and does not show that discounts caused higher revenue.

**Recommended actions**

- Plan extra stock and staffing for the late-year and weekend peak, then check whether the pattern repeats in the next period.
- Ask Pune what supports its stronger Beverages revenue and run a measured one-month test at another outlet.

> Edit this draft to match the findings you want to emphasize. Keep each claim traceable to a table or chart.


## 11. Worked answers and interpretation notes

These concise notes match the guide's exercises. They are repeated here so the notebook works as a complete reference after you try the prompts.

In [ ]:
# First look
print("Rows and columns:", cafe.shape)
print("Date range:", cafe["Date"].min().date(), "to", cafe["Date"].max().date())
print("Missing cells:", int(cafe.isna().sum().sum()))
print("Outlet counts:", cafe["Outlet"].value_counts().sort_index().to_dict())

# Item growth and merchandise weekday lift
print("Feb-to-Dec item growth (%):")
display(((by_item_month.loc[12] / by_item_month.loc[2] - 1) * 100).round(1))
print("Merchandise weekend lift (%):", round(merch_weekend_lift * 100, 1))

# Month/outlet high cell and offers for beverages
print("Highest month/outlet cell:", highest_month_outlet, round(grid_month_outlet.stack().max(), 1))
print("Beverages offer averages:")
display(bev_offer.round(1))
print("Beverages offer step changes:")
display(bev_offer.diff().round(1))

# Extreme values and blunt outlet rule
print("Item rule removals:", len(flagged))
print("Outlet rule removals:", int(too_many_mask.sum()))


### Interpretation guide

- The table has 3,285 rows, covers 1 January to 31 December 2025, and has no missing values. Each outlet has 1,095 rows because every outlet has a row for each item on each day.
- December is the highest month. Merchandise grows most from February to December in percentage terms, though all three items follow a similar seasonal pattern. Merchandise's weekend lift is about 35%, close to the all-item lift.
- Pune's December cell is the highest Month × Outlet average (about 403 Rs thousand). The outlet order is not identical in every month; inspect the February exception before claiming a fixed ranking.
- For Beverages, average revenue is highest at a 25% offer and rises after a small 0% to 5% dip. This remains association, not proof that the offer caused the revenue difference.
- The item-specific rule removes exactly three injected slips. The outlet-level rule flags seven rows, including normal high Beverages days; grouping by item compares more similar records.
- Under all app filters, the cleaned view has 3,282 rows and total revenue of about Rs 920.6 million. Pune in months 11–12 has 182 rows and about Rs 66.2 million.


## 12. Final checklist

Before sharing your work, confirm that charts have titles and units, your outlier decision is written down, each report claim includes a number, and the HTML dashboard opens with hover/zoom interactions. Restart the kernel and run all cells from the top when you are ready to use the notebook as a reproducible analysis.